In [ ]:
import cv2
import mediapipe as mp
import numpy as np
import os
import time
from datetime import datetime
import math


# Application configuration and global constants
class Config:
    WIDTH, HEIGHT = 1280, 720
    APP_NAME = "Virtual Painter Pro - Thesis Edition"

    OUTPUT_FOLDER = "Artworks_Gallery"
    VIDEO_FOLDER = os.path.join(OUTPUT_FOLDER, "Videos")

    LEFT_BAR_W = 175
    RIGHT_BAR_W = 380

    BTN_H = 44
    BTN_GAP = 10
    PAD = 12
    DOCK_GAP = 8

    TITLE_H = 22
    SECTION_GAP = 12

    SMOOTHING_FACTOR = 0.20

    MAX_HISTORY = 10

    BRUSH_MIN, BRUSH_MAX = 3, 50
    ERASER_MIN, ERASER_MAX = 20, 140

    VIDEO_FPS = 30.0

    FILL_MIN_AREA = 30

    MANDALA_COUNTS = [6, 8, 10, 12, 16]
    MANDALA_MIRROR = True

    COLORS = {
        "NERO": (0, 0, 0),
        "BIANCO": (255, 255, 255),
        "ROSSO": (0, 0, 255),
        "VERDE": (0, 255, 0),
        "BLU": (255, 0, 0),
        "GIALLO": (0, 255, 255),
        "VIOLA": (255, 0, 255),
        "ARANCIO": (0, 69, 255),
        "CIANO": (255, 255, 0)
    }


# Clamp a value inside a given range
def clamp(v, vmin, vmax):
    return max(vmin, min(vmax, v))


# Draw a UI panel with background and border
def draw_panel(img, x0, y0, x1, y1, bg=(24, 24, 24), border=(80, 80, 80)):
    cv2.rectangle(img, (x0, y0), (x1, y1), bg, -1)
    cv2.rectangle(img, (x0, y0), (x1, y1), border, 1)


# Draw a section title on screen
def draw_title(img, text, x, y):
    cv2.putText(
        img, text, (x, y),
        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (230, 230, 230), 2
    )


# Basic UI button abstraction
class UIButton:
    def __init__(self, name, rect, bgr, kind="ACTION", label=None, sublabel=None):
        self.name = name
        self.rect = rect
        self.bgr = bgr
        self.kind = kind
        self.label = label if label else name
        self.sublabel = sublabel

    # Check if a point is inside the button area
    def hit(self, x, y):
        bx, by, bw, bh = self.rect
        return bx <= x <= bx + bw and by <= y <= by + bh

    # Render the button on screen
    def draw(self, img, selected=False, label_override=None, sublabel_override=None):
        bx, by, bw, bh = self.rect

        overlay = img.copy()
        cv2.rectangle(overlay, (bx, by), (bx + bw, by + bh), (45, 45, 45), -1)
        cv2.addWeighted(overlay, 0.55, img, 0.45, 0, img)

        cv2.rectangle(
            img,
            (bx + 5, by + 5),
            (bx + bw - 5, by + bh - 5),
            self.bgr,
            -1
        )

        border = (255, 255, 255) if selected else (100, 100, 100)
        cv2.rectangle(img, (bx, by), (bx + bw, by + bh),
                      border, 2 if selected else 1)

        label = label_override if label_override else self.label
        cv2.putText(
            img, label, (bx + 10, by + 28),
            cv2.FONT_HERSHEY_PLAIN, 1.2, (255, 255, 255), 2
        )

        sub = sublabel_override if sublabel_override else self.sublabel
        if sub:
            cv2.putText(
                img, sub, (bx + 10, by + bh - 8),
                cv2.FONT_HERSHEY_PLAIN, 1.0, (235, 235, 235), 1
            )


# Hand tracking wrapper using MediaPipe
class HandDetector:
    def __init__(self, max_hands=1, detection_con=0.8, track_con=0.5):
        self.mpHands = mp.solutions.hands
        self.hands = self.mpHands.Hands(
            static_image_mode=False,
            max_num_hands=max_hands,
            min_detection_confidence=detection_con,
            min_tracking_confidence=track_con
        )
        self.mpDraw = mp.solutions.drawing_utils
        self.tipIds = [4, 8, 12, 16, 20]
        self.results = None
        self.lmList = []

    # Detect hands and optionally draw landmarks
    def find_hands(self, img, draw=True):
        imgRGB = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        self.results = self.hands.process(imgRGB)
        if draw and self.results.multi_hand_landmarks:
            for handLms in self.results.multi_hand_landmarks:
                self.mpDraw.draw_landmarks(
                    img, handLms, self.mpHands.HAND_CONNECTIONS
                )
        return img

    # Extract landmark pixel positions
    def find_position(self, img):
        self.lmList = []
        if self.results and self.results.multi_hand_landmarks:
            h, w, _ = img.shape
            for idx, lm in enumerate(self.results.multi_hand_landmarks[0].landmark):
                cx, cy = int(lm.x * w), int(lm.y * h)
                self.lmList.append([idx, cx, cy])
        return self.lmList

    # Determine which fingers are raised
    def fingers_up(self):
        if not self.lmList:
            return []

        fingers = []
        fingers.append(1 if self.lmList[4][1] < self.lmList[3][1] else 0)
        for i in range(1, 5):
            fingers.append(
                1 if self.lmList[self.tipIds[i]][2] <
                self.lmList[self.tipIds[i] - 2][2] else 0
            )
        return fingers

    # Compute distance between two landmarks
    def distance(self, p1, p2):
        x1, y1 = self.lmList[p1][1], self.lmList[p1][2]
        x2, y2 = self.lmList[p2][1], self.lmList[p2][2]
        return ((x2 - x1) ** 2 + (y2 - y1) ** 2) ** 0.5


# Main painting engine
class PainterEngine:
    def __init__(self):
        self.img_canvas = np.zeros((Config.HEIGHT, Config.WIDTH, 3), np.uint8)
        self.paint_mask = np.zeros((Config.HEIGHT, Config.WIDTH), np.uint8)

        self.undo_stack = []
        self.redo_stack = []

        self.brush_color = Config.COLORS["ROSSO"]
        self.brush_size = 15
        self.eraser_size = 60
        self.is_eraser = False

        self.xp, self.yp = 0, 0
        self.cx, self.cy = 0, 0

        self.stroke_style = "SOLID"
        self.style_names = [
            "SOLID", "DOTTED", "DASH", "SPRAY",
            "MARKER", "NEON", "CHALK"
        ]

        if not os.path.exists(Config.OUTPUT_FOLDER):
            os.makedirs(Config.OUTPUT_FOLDER)
        if not os.path.exists(Config.VIDEO_FOLDER):
            os.makedirs(Config.VIDEO_FOLDER)


# Main application loop
def main():
    cap = cv2.VideoCapture(0)
    cap.set(3, Config.WIDTH)
    cap.set(4, Config.HEIGHT)

    detector = HandDetector()
    engine = PainterEngine()

    while True:
        success, frame = cap.read()
        if not success:
            break

        frame = cv2.flip(frame, 1)
        frame = detector.find_hands(frame)
        _ = detector.find_position(frame)

        cv2.imshow(Config.APP_NAME, frame)

        if cv2.waitKey(1) & 0xFF == 27:
            break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()